# 04 - Insights, Validation & Recommendations
## Global Superstore Dataset (2011-2014)

تحويل نتائج EDA والـ KPIs لرؤى منظمة، تحدّيها بأسئلة نقدية، وتحويلها لتوصيات عملية.

In [ ]:
import pandas as pd
df = pd.read_csv('../data/processed/cleaned_superstore.csv', parse_dates=['Order Date', 'Ship Date'])
df.head()

## Insight 1: Discount Threshold at 20%

**What happened:** Average profit per order is positive up to a 20% discount,
then turns consistently negative beyond it.

**Evidence:**

In [ ]:
bins = [-0.01,0,0.1,0.2,0.3,0.4,0.5,0.6,0.7,0.85]
labels = ['0%','0-10%','10-20%','20-30%','30-40%','40-50%','50-60%','60-70%','70-85%']
df['Discount_Bucket'] = pd.cut(df['Discount'], bins=bins, labels=labels)
print(df.groupby('Discount_Bucket', observed=True)['Profit'].mean().round(1))

### Validation
- Sample size per bucket: كافي (أصغر مجموعة 318 طلب)
- مش نتيجة Outliers قليلة (موزع على آلاف الطلبات)
- **تحفظ:** Correlation مش Causation مؤكدة 100% (مفيش عمود تكلفة أصلية نفصل بيه الأثرين)

In [ ]:
high_disc = df[df['Discount'] > 0.20]
losing_orders = high_disc[high_disc['Profit'] < 0]
print('High-discount orders:', len(high_disc), f"({len(high_disc)/len(df)*100:.1f}% of all orders)")
print('Losing money:', len(losing_orders), f"({len(losing_orders)/len(high_disc)*100:.1f}% of the group)")
print('Net profit of this group:', round(high_disc['Profit'].sum(),0))

## Insight 2: Tables — Discount-Driven Loss

**Evidence:**

In [ ]:
tables = df[df['Sub-Category']=='Tables']
tables_low = tables[tables['Discount'] <= 0.2]['Profit'].sum()
tables_high = tables[tables['Discount'] > 0.2]['Profit'].sum()
print(f'Tables profit (discount <=20%): ${tables_low:,.0f}')
print(f'Tables profit (discount >20%):  ${tables_high:,.0f}')
print(f'Total Tables profit: ${tables["Profit"].sum():,.0f}')

### Validation: Is the loss driven by a handful of outliers?

In [ ]:
worst10 = tables.nsmallest(10, 'Profit')
print(worst10[['Market','Discount','Sales','Profit']])
print()
pct = worst10['Profit'].sum() / tables['Profit'].sum() * 100
print(f'Worst 10 orders account for only {pct:.1f}% of total Tables loss -> loss is broadly distributed, not a few outliers')

## Insight 3: EMEA — Systemic Pricing Issue

**Evidence:**

In [ ]:
emea = df[df['Market']=='EMEA']
ecat = emea.groupby('Category').agg(Sales=('Sales','sum'), Profit=('Profit','sum'))
ecat['Margin_%'] = (ecat['Profit']/ecat['Sales']*100).round(1)
print(ecat)
print()
print(df.groupby('Market')['Discount'].mean().round(3).sort_values(ascending=False))

### Validation
Canada excluded from 'best market' comparisons (tiny sample: only $66,928 in sales).
Other potential drivers (e.g., regional operating costs) aren't in this dataset — flagged as a limitation.

## Recommendations Summary

| # | Recommendation | Evidence | Expected Impact |
|---|---|---|---|
| 1 | Global discount cap at 20% + approval workflow for exceptions | 89.9% of >20%-discount orders lose money; -$814,682 net | Recover a meaningful share of the -$814,682 annual loss |
| 2 | Dedicated 20% discount cap for Tables | Tables profitable (+$68,458) below 20% discount, loses (-$132,541) above it | Flip Tables from -$64,083 to net positive |
| 3 | Review EMEA pricing/negotiation policy vs. EU benchmark | EMEA avg discount 19.6% vs EU 10.3%; margin gap ~6-7 points | Potential margin improvement of several percentage points |
| 4 | Continue current growth strategy | CAGR 23.9%, stable ~11-12% margin | Maintain healthy growth |

**No action needed:** Ship Mode / Processing Time (correlation with profit ~0 — investigated and ruled out).